# We should take a look to see how our signal actually looks a like

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import random

from gyroscope.config import PROCESSED_DATA_DIR


def plot_raw_data(processed_data_dir):
    """Find all X_*.npy files, loads them along wit their y_*.npy labels and plot them"""

    # Find all X arrays
    x_files = glob.glob(os.path.join(processed_data_dir, 'x_*.npy'))

    if not x_files:
        print(f"No X_*.npy files found in {processed_data_dir}")
        return
    
    # iterate through each dataset
    for x_path in x_files:

        # Extract the dataset name
        filename = os.path.basename(x_path)
        dataset_name = filename.replace('X_', '').replace('npy', '')

        # Construct the path to the matching y labels
        y_path = os.path.join(processed_data_dir, f'y_{dataset_name}npy')

        if not os.path.exists(y_path):
            print(f"warning, found {filename} but no matching labels: {os.path.basename(y_path)}, skipping ..")
            continue

        # Load the numpy 
        X = np.load(x_path)
        y = np.load(y_path)

        # Separate the indices for good and wrong
        normal_idx = np.where(y == 0)[0]
        anomaly_idx = np.where(y ==1)[0]

        plt.figure(figsize=(12, 4))
        plt.title(f"Dataset: {dataset_name} | Total samples {len(y)} | Normal: {len(normal_idx)} | Anomaly: {len(anomaly_idx)}")
        plt.xlabel("Time step index")
        plt.ylabel("Senor ampltiude")

        # Plot a normal signal
        sampled_normal = random.sample(list(normal_idx), k=1)
        for i, idx in enumerate(sampled_normal):
            label = 'Normal' if i == 0 else None
            plt.plot(X[idx], color='blue', label=label)

        # Plot a random anomaly signal
        sampled_anomaly = random.sample(list(anomaly_idx), k=1)
        for i, idx in enumerate(sampled_anomaly):
            label = 'Anomaly' if i == 0 else None
            plt.plot(X[idx], color='red', label=label)
        
        plt.legend()
        plt.tight_layout()
        plt.show()

### Lets plot some samples from the data

In [ ]:
plot_raw_data(processed_data_dir=PROCESSED_DATA_DIR)

### Lets see also how would look like a signals from csv files

In [ ]:
import pandas as pd
import plotly.express as px
import os

from gyroscope.config import RAW_DATA_DIR

def plot_interactive_raw_file(file_path):
    """
    Loads a raw csv file and creates interactive plotly graph 
    """
    print(f"Loading data from: {file_path}")

    # load the data
    df = pd.read_csv(file_path, skiprows=1, sep=r'\s+', names=['Timestamp', 'Sensor_Amplitude'])

    # Convert timestamp to real Datetime object
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])

    print(f"Successfully loaded {len(df)} rows")

    # Create interactive line chart using the real time
    fig = px.line(data_frame=df, x='Timestamp', y='Sensor_Amplitude', title=f"Raw singla {os.path.basename(file_path)}", labels={"Sensor_Amplitude": "Amplitude (rad/sec)"})

    # Add range slider
    fig.update_layout(xaxis=dict(rangeslider=dict(visible=True), type="date"),template="plotly_white", hovermode="x unified")

    fig.show()

### Lets load first csv file

In [ ]:
plot_interactive_raw_file(os.path.join(RAW_DATA_DIR, "1.csv"))

#### Lets load second csv file 

In [ ]:
plot_interactive_raw_file(os.path.join(RAW_DATA_DIR, "3.csv"))